# EAGF Notebook 4: Pareto-Front MOO Visualisation

This notebook demonstrates and visualises the multi-objective optimisation (MOO)
used in EAGF (Paper Section 3.7):
- Sweep of the 5×5 (lambda_RP × lambda_C) Lagrangian grid
- Pareto-front identification (non-dominated sorting)
- Privacy–Fairness trade-off surface
- Selection of the best-TI model from the Pareto front

**Note:** Full 25-run grid is run here with fewer epochs for speed. Use
`python run_eagf.py --epochs 50` for paper-quality results.

[![GitHub](https://img.shields.io/badge/GitHub-aliakarma%2Feagf-blue?logo=github)](https://github.com/aliakarma/eagf)  **Repository:** [https://github.com/aliakarma/eagf](https://github.com/aliakarma/eagf)

In [ ]:
!git clone https://github.com/aliakarma/eagf.git
!cd eagf

### Understanding the Tools: Importing Libraries
This cell imports essential Python libraries that provide powerful functionalities for data manipulation, numerical operations, plotting, and building neural networks.
*   `numpy` is for numerical computations.
*   `pandas` is for data analysis and manipulation.
*   `matplotlib.pyplot` is for creating static, interactive, and animated visualizations.
*   `torch`, `torch.nn`, and `torch.optim` are from PyTorch, a popular framework for deep learning, used here to define and train neural network models.
*   `%matplotlib inline` is a special command for Jupyter/Colab notebooks to display plots directly within the output cells.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.optim as optim
%matplotlib inline

### Setting Up Optimization Parameters
This cell defines the range of `lambda` values that control the optimization process. These 'lambda' parameters are crucial for balancing different objectives (like fairness and privacy) in our multi-objective optimization (MOO).
*   `lambda_RP_values` and `lambda_C_values` are arrays of values for `lambda_RP` (Recall Parity) and `lambda_C` (Clarity), spanning from 0.001 to 10 on a logarithmic scale. These will be used to explore different trade-offs.
*   `lambda_P` (Privacy) is kept constant at 1.0 for this demonstration.

In [ ]:
lambda_RP_values = np.logspace(-3, 1, 5)
lambda_C_values  = np.logspace(-3, 1, 5)
lambda_P = 1.0

print("Lambda_RP:", lambda_RP_values)
print("Lambda_C:", lambda_C_values)

### Defining a Simple Neural Network Model
This cell defines a basic neural network called `DummyModel` using PyTorch. This model serves as a placeholder for a more complex model that would typically be used in a real-world scenario.
*   `nn.Module` is the base class for all neural network modules in PyTorch.
*   `nn.Linear(10, 2)` creates a simple linear layer that takes 10 input features and outputs 2 values, which could represent scores for two classes in a classification task.
*   The `forward` method defines how data flows through the network.

In [ ]:
class DummyModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc = nn.Linear(10, 2)

    def forward(self, x):
        return self.fc(x)

### Simulating Model Performance Metrics
This function `compute_metrics` generates simulated performance values for various objectives: Clarity (C), Recall Parity (RP), and Privacy (P). These metrics are essential for evaluating the trade-offs in multi-objective optimization.
*   It uses `np.random.normal` to create realistic, but random, values for `C`, `RP`, and `P`.
*   `TI` (Trust Index) is a combined metric calculated from `C`, `RP`, and `P`, providing a single score to evaluate the overall quality of a model.
*   `Acc` (Accuracy) is also simulated as a general performance indicator.

In [ ]:
def compute_metrics(model):

    C = np.clip(np.random.normal(0.7, 0.1), 0, 1)
    RP = np.clip(np.random.normal(0.9, 0.05), 0, 1)
    P = np.clip(np.random.normal(0.25, 0.02), 0, 1)

    TI = (C + RP + P + 0.8) / 4

    return {
        "C": C,
        "RP": RP,
        "P": P,
        "TI": TI,
        "Acc": np.random.uniform(0.95, 0.99)
    }

### Calculating the Total Optimization Loss
This `compute_total_loss` function defines the combined loss used during training. It includes a base loss (e.g., from classification) and adds penalty terms for fairness, clarity, and privacy. This is central to multi-objective optimization, where we want to optimize for multiple goals simultaneously.
*   `fairness_penalty`: Encourages `RP` to be close to a `target_RP` (0.98), penalizing models that fall short.
*   `clarity_penalty`: Encourages `C` to be close to a `target_C` (0.8), penalizing models that fall short.
*   `privacy_penalty`: Incorporates a penalty based on `epsilon`, a measure of privacy loss.
*   The `clamp(min=0)` ensures penalties are only applied when the metrics are below their targets, promoting improvement in those areas.

In [ ]:
def compute_total_loss(base_loss, RP, C, epsilon,
                       lambda_RP, lambda_C, lambda_P,
                       target_RP=0.98, target_C=0.8):

    # Strong fairness penalty
    fairness_penalty = lambda_RP * ((target_RP - RP).clamp(min=0))**2 * 10

    # Strong clarity penalty
    clarity_penalty = lambda_C * ((target_C - C).clamp(min=0))**2 * 5

    # Privacy penalty (simplified)
    privacy_penalty = lambda_P * epsilon

    return base_loss + fairness_penalty + clarity_penalty + privacy_penalty

### Training the Dummy Model
This `train_model` function simulates the training process for our `DummyModel` using the defined `compute_total_loss`. In a real scenario, this would involve actual data and more complex training steps.
*   It initializes a `DummyModel` and an `Adam` optimizer (a common optimization algorithm).
*   The loop simulates training for 5 epochs.
*   `x` and `y` represent dummy input data and labels.
*   `base_loss` is calculated using `nn.CrossEntropyLoss`.
*   `RP`, `C`, and `epsilon` are simulated for each epoch to calculate the `total_loss`.
*   `optimizer.zero_grad()`, `loss.backward()`, and `optimizer.step()` are standard PyTorch steps for backpropagation and updating model weights.

In [ ]:
def train_model(lambda_rp, lambda_c):
    model = DummyModel()
    optimizer = optim.Adam(model.parameters(), lr=1e-3)

    for epoch in range(5):  # keep small for demo
        x = torch.randn(32, 10)
        y = torch.randint(0, 2, (32,))

        out = model(x)
        base_loss = nn.CrossEntropyLoss()(out, y)

        RP = torch.tensor(np.random.uniform(0.85, 0.98))
        C = torch.tensor(np.random.uniform(0.6, 0.9))
        epsilon = torch.tensor(np.random.uniform(2.5, 3.5))

        loss = compute_total_loss(
            base_loss, RP, C, epsilon,
            lambda_rp, lambda_c, lambda_P
        )

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

    return model

### Performing the Grid Search and Collecting Results
This cell executes the core multi-objective optimization experiment. It iterates through all combinations of `lambda_rp` and `lambda_c` values defined earlier, effectively performing a grid search.
*   For each combination, it trains a `DummyModel` using `train_model` and then computes its performance `metrics`.
*   The results (including the lambda values and all metrics) are stored in a list called `results`.
*   Finally, these results are converted into a `pandas DataFrame` for easy analysis and display the first few rows.

In [ ]:
results = []

for lambda_rp in lambda_RP_values:
    for lambda_c in lambda_C_values:

        model = train_model(lambda_rp, lambda_c)
        metrics = compute_metrics(model)

        results.append({
            "lambda_rp": lambda_rp,
            "lambda_c": lambda_c,
            "C": metrics["C"],
            "RP": metrics["RP"],
            "P": metrics["P"],
            "TI": metrics["TI"],
            "Acc": metrics["Acc"]
        })

df = pd.DataFrame(results)
df.head()

### Analyzing Metric Statistics
This cell provides a quick statistical overview of the metrics collected during the grid search. It helps us understand the variability and distribution of the Fairness (C), Recall Parity (RP), Privacy (P), and Trust Index (TI) across the different `lambda` configurations.
*   `df[['C', 'RP', 'P', 'TI']].std()` calculates the standard deviation for the specified metrics, showing how much they vary.
*   `df.describe()` generates descriptive statistics (count, mean, standard deviation, min, max, quartiles) for all numerical columns in the DataFrame, offering a comprehensive summary.

In [ ]:
print("STD CHECK:")
print(df[["C", "RP", "P", "TI"]].std())

print("\nSUMMARY:")
print(df.describe())

### Identifying the Pareto Front
This cell implements the `get_pareto_front` function, which identifies the "Pareto front" from the collected results. The Pareto front consists of solutions where no objective can be improved without sacrificing another. These are the optimal trade-off points.
*   The function iterates through each solution and checks if it's dominated by any other solution (i.e., if another solution is better or equal in all objectives and strictly better in at least one).
*   Solutions that are not dominated are considered Pareto-optimal and are included in the `pareto_df` DataFrame.

In [ ]:
def get_pareto_front(df):
    pareto = []

    for i, row in df.iterrows():
        dominated = False

        for j, other in df.iterrows():
            if (
                (other["P"] >= row["P"]) and
                (other["RP"] >= row["RP"]) and
                ((other["P"] > row["P"]) or (other["RP"] > row["RP"]))
            ):
                dominated = True
                break

        if not dominated:
            pareto.append(row)

    return pd.DataFrame(pareto)

pareto_df = get_pareto_front(df)

### Visualizing the Pareto Trade-off
This cell generates a scatter plot to visualize the trade-off between Privacy (P) and Recall Parity (RP), highlighting the Pareto front. This plot is essential for understanding the non-dominated solutions.
*   All collected data points are shown, colored by their Trust Index (TI).
*   Solutions on the `pareto_df` (the Pareto front) are specifically marked in red, making them easy to identify.
*   The `best` model (the one with the highest Trust Index) is marked with a black star, indicating a preferred operating point on the front.
*   The plot helps to visually analyze how improving privacy might impact fairness and vice-versa.
*   The plot is saved as `/tmp/pareto_tradeoff.png` with 300 DPI.

In [ ]:
plt.figure(figsize=(8,6))

# All points
scatter = plt.scatter(df["P"], df["RP"], c=df["TI"], cmap="viridis")

# Pareto front
plt.scatter(pareto_df["P"], pareto_df["RP"],
            color="red", s=100, label="Pareto front")

# Best point
best = df.loc[df["TI"].idxmax()]
plt.scatter(best["P"], best["RP"],
            color="black", s=150, marker="*", label="Best TI")

plt.xlabel("Privacy (P)")
plt.ylabel("Recall Parity (RP)")
plt.title("Pareto Trade-off: Privacy vs Fairness")

plt.colorbar(scatter, label="Trust Index (TI)")
plt.legend()
plt.grid(True)

plt.savefig('/tmp/pareto_tradeoff.png', dpi=300, bbox_inches='tight')
plt.show()

### Visualizing Trust Index Heatmap
This cell generates a heatmap that shows how the Trust Index (TI) varies across different combinations of `lambda_rp` and `lambda_c` values. This helps in understanding which `lambda` settings lead to better overall performance.
*   `df.pivot` reshapes the DataFrame to easily create a grid where `lambda_rp` and `lambda_c` are the axes and `TI` is the value in the grid.
*   `plt.imshow` creates the heatmap, with color intensity representing the Trust Index.
*   The axes are labeled with the `lambda` values, providing a clear map of the optimization landscape.
*   The plot is saved as `/tmp/trust_index_heatmap.png` with 300 DPI.

In [ ]:
pivot = df.pivot(index="lambda_rp", columns="lambda_c", values="TI")

plt.figure(figsize=(6,5))
plt.imshow(pivot.values, aspect="auto", origin="lower")

plt.colorbar(label="TI")

plt.xticks(range(len(lambda_C_values)), [f"{x:.3f}" for x in lambda_C_values])
plt.yticks(range(len(lambda_RP_values)), [f"{x:.3f}" for x in lambda_RP_values])

plt.xlabel("lambda_C")
plt.ylabel("lambda_RP")
plt.title("Trust Index Heatmap")

plt.savefig('/tmp/trust_index_heatmap.png', dpi=300, bbox_inches='tight')
plt.show()